# EYEWAZ — train your Urdu Piper voice

**Before running:** Runtime → Change runtime type → **GPU (T4)**. Then upload
`dataset-female.zip` via the Files panel (📁 on the left). Then Runtime → **Run all**.

Repeat later with `dataset-male.zip` by setting `VOICE = "male"` in cell 3.


### 1. Check the GPU is on


In [ ]:
!nvidia-smi -L || print('NO GPU — set Runtime > Change runtime type > GPU first!')


### 2. Install Piper training


In [ ]:
!git clone -q https://github.com/rhasspy/piper
%cd piper/src/python
!pip install -q -e .
!pip install -q -r requirements.txt
!bash build_monotonic_align.sh
print('Piper training installed.')


### 3. Settings + unzip your dataset
Make sure `dataset-female.zip` is uploaded to `/content/` first (Files panel).


In [ ]:
import os
VOICE = 'female'        # set to 'male' for the second voice
DATA  = f'/content/dataset-{VOICE}'
OUT   = f'/content/train-{VOICE}'
assert os.path.exists(f'/content/dataset-{VOICE}.zip'), 'Upload dataset-%s.zip to /content first' % VOICE
!unzip -oq /content/dataset-{VOICE}.zip -d /content/
print('clips:', len(os.listdir(f'{DATA}/wav')))


### 4. Preprocess (Urdu phonemes via espeak-ng)


In [ ]:
!python -m piper_train.preprocess \
  --language ur \
  --input-dir {DATA} \
  --output-dir {OUT} \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050


### 5. Get a pretrained checkpoint to fine-tune from
Fine-tuning from a medium 22.05 kHz checkpoint gives a usable voice from far
less data than training from scratch. We use the public Lessac (en_US) medium
checkpoint — the acoustic model transfers; Urdu phonemes come from espeak-ng.


In [ ]:
# Lessac medium training checkpoint (~). If this URL 404s, browse
# https://huggingface.co/datasets/rhasspy/piper-checkpoints and grab any */medium/*.ckpt
URL='https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
!wget -q --show-progress -O /content/pretrained.ckpt "$URL"
import os; print('checkpoint MB:', round(os.path.getsize('/content/pretrained.ckpt')/1e6,1))


### 6. Train
Watch the audio samples it logs under the ▶ outputs. With ~6 min of data this
is a **pilot** — it'll sound like you but rough; record more (30–60 min) for a
quality voice. **Stop the cell** (■) when samples sound good enough.


In [ ]:
!python -m piper_train \
  --dataset-dir {OUT} \
  --accelerator gpu --devices 1 \
  --batch-size 16 \
  --max_epochs 4000 \
  --checkpoint-epochs 5 \
  --quality medium \
  --resume_from_checkpoint /content/pretrained.ckpt


### 7. Export to ONNX (this is what EYEWAZ runs)


In [ ]:
ck = !ls -t {OUT}/lightning_logs/version_0/checkpoints/*.ckpt | head -1
ck = ck[0]; print('exporting', ck)
!python -m piper_train.export_onnx {ck} /content/eyewaz-urdu-{VOICE}.onnx
!cp {OUT}/config.json /content/eyewaz-urdu-{VOICE}.onnx.json
from google.colab import files
files.download(f'/content/eyewaz-urdu-{VOICE}.onnx')
files.download(f'/content/eyewaz-urdu-{VOICE}.onnx.json')


### Done
You now have `eyewaz-urdu-female.onnx` + `.onnx.json`. Drop both into
`tts-local/` (or send them to me) and I'll wire the voice into every EYEWAZ
surface — local server, NVDA, JAWS/SAPI, Chrome, Android, and the cloud service.
